In [ ]:
import pandas as pd
import pingouin as pg
import numpy as np
from scipy import stats

input_df = pd.read_pickle(snakemake.input[0])

In [ ]:
region_list = [
    'Left-Cerebral-White-Matter', 'Right-Cerebral-White-Matter', 
    'frontal_left_wm', 'frontal_right_wm', 
    'temporal_left_wm', 'temporal_right_wm',
    'parietal_left_wm', 'parietal_right_wm',
    'occipital_left_wm', 'occipital_right_wm',
    'cerebellum_left', 'cerebellum_right'
              ]
filtered_df = input_df.query('region in @region_list and outliers_removed == True and segmentation == "mni_icbm152_nlin_asym_09c_wm90percent_lobes"')
filtered_df['b1corr'] = filtered_df['contrast'].str.contains('b1corr')
filtered_df['contrast'] = filtered_df['contrast'].str.replace('_b1corr', '')
filtered_df['Vref_lessthan100'] = filtered_df['acquisition'].str.contains("(?i)Vref(0*(?:[1-9][0-9]?|100))", regex=True)
filtered_df['acquisition'] = filtered_df['acquisition'].str.replace("(?i)Vref(0*(?:[1-9][0-9]?|100))", "", regex=True)

In [ ]:
df_ihMTv4BPsagHSarp3intlin3dumFoV288 = filtered_df[filtered_df['acquisition'].str.contains("ihMTv4BPsagHSarp3intlin3dumFoV288")]

In [ ]:
icc_vref = filtered_df.groupby(
    ['subject', 'session', 'field_strength', 'modality', 'contrast', 'b1corr', 'acquisition']).apply(
    lambda d: pg.intraclass_corr(
        data=d, targets='region', raters='Vref_lessthan100', ratings='mean'
    )).query('Type == "ICC(C,k)"').reset_index()
icc_vref = icc_vref.rename({'CI95':'ICC_CI95'}, axis=1)[[
    'subject', 'session', 'field_strength', 'modality', 'contrast', 'b1corr', 'acquisition', 'ICC', 'ICC_CI95']]
icc_vref['test'] = "Vref"
icc_vref

In [ ]:
icc_vref[~icc_vref.isin([np.inf, -np.inf, np.nan])].dropna()

In [ ]:
df_ihMTv4BPsagHSarp3intlin3dumFoV288 = df_ihMTv4BPsagHSarp3intlin3dumFoV288[[
    'subject', 'session', 'field_strength', 'modality', 'contrast', 'b1corr', 'acquisition', 'region', 'mean'
]]

In [ ]:
df_ihMTv4BPsagHSarp3intlin3dumFoV288Vref100 = df_ihMTv4BPsagHSarp3intlin3dumFoV288.query(
    'acquisition == "ihMTv4BPsagHSarp3intlin3dumFoV288"'
).rename(
    {'mean': 'ihMTv4BPsagHSarp3intlin3dumFoV288Vref100'}, axis=1
).drop(
    'acquisition', axis=1
).reset_index(drop=True)

df_ihMTv4BPsagHSarp3intlin3dumFoV288Vref90 = df_ihMTv4BPsagHSarp3intlin3dumFoV288.query(
    'acquisition == "ihMTv4BPsagHSarp3intlin3dumFoV288Vref90"'
).rename(
    {'mean': 'ihMTv4BPsagHSarp3intlin3dumFoV288Vref90'}, axis=1
).drop(
    'acquisition', axis=1
).reset_index(drop=True)

keys = ["subject", "session", "field_strength", "modality", "contrast", "b1corr", "region"]
df_ihMTv4BPsagHSarp3intlin3dumFoV288_means_by_vref = df_ihMTv4BPsagHSarp3intlin3dumFoV288Vref100.merge(
    df_ihMTv4BPsagHSarp3intlin3dumFoV288Vref90,
    on=keys,
    how="inner",
    suffixes=("_Vref100", "_Vref90")
)

In [ ]:
pcc_ihMTv4BPsagHSarp3intlin3dumFoV288_100vs90 = df_ihMTv4BPsagHSarp3intlin3dumFoV288_means_by_vref.groupby(
    ['subject', 'session', 'field_strength', 'modality', 'contrast', 'b1corr']).apply(
    lambda d: pg.pairwise_corr(
        data=d, columns=['ihMTv4BPsagHSarp3intlin3dumFoV288Vref100', 'ihMTv4BPsagHSarp3intlin3dumFoV288Vref90']
    )).reset_index()
pcc_ihMTv4BPsagHSarp3intlin3dumFoV288_100vs90 = pcc_ihMTv4BPsagHSarp3intlin3dumFoV288_100vs90.rename(
    {'r':'pearson_r', 'CI95':'pearson_CI95'}, axis=1)[[
    'subject', 'session', 'field_strength', 'modality', 'contrast', 'b1corr', 'pearson_r', 'pearson_CI95'
]]
pcc_ihMTv4BPsagHSarp3intlin3dumFoV288_100vs90['test'] = "ihMTv4BPsagHSarp3intlin3dumFoV288 Vref100 vs Vref90"
pcc_ihMTv4BPsagHSarp3intlin3dumFoV288_100vs90

In [ ]:
icc_ihMTv4BPsagHSarp3intlin3dumFoV288Vref100_corrvsuncorr = df_ihMTv4BPsagHSarp3intlin3dumFoV288Vref100.groupby(
    ['subject', 'session', 'field_strength', 'modality', 'contrast']).apply(
    lambda d: pg.intraclass_corr(
        data=d, targets='region', raters='b1corr', ratings='ihMTv4BPsagHSarp3intlin3dumFoV288Vref100'
    )).query('Type == "ICC(C,k)"').reset_index()
# icc_ihMTv4BPsagHSarp3intlin3dumFoV288Vref100_corrvsuncorr = icc_ihMTv4BPsagHSarp3intlin3dumFoV288Vref100_corrvsuncorr.rename(
#     {'CI95':'ICC_CI95'}, axis=1)[[
#     'subject', 'session', 'field_strength', 'modality', 'contrast', 'b1corr', 'ICC', 'ICC_CI95']]
icc_ihMTv4BPsagHSarp3intlin3dumFoV288Vref100_corrvsuncorr['test'] = "ihMTv4BPsagHSarp3intlin3dumFoV288Vref100 b1corr vs uncorr"
icc_ihMTv4BPsagHSarp3intlin3dumFoV288Vref100_corrvsuncorr

In [ ]:
df_ihMTv4BPsagHSarp3intlin3dumFoV288Vref100_corr = df_ihMTv4BPsagHSarp3intlin3dumFoV288Vref100.query(
    'b1corr == True').rename(
    {'ihMTv4BPsagHSarp3intlin3dumFoV288Vref100':'b1corr'}, axis=1
).drop(
    'acquisition', axis=1
).reset_index(drop=True)

In [ ]:
results = pd.merge(icc_ihMTv4BPsagHSarp3intlin3dumFoV288_100vs90, pcc_ihMTv4BPsagHSarp3intlin3dumFoV288_100vs90)
results.insert(0, 'test', results.pop('test'))
results